## **Scratchpad**


- Survey is Mostly Qualitative
    - Either categorical / rank scale (1-5)
- So many questions that can be asked / answered
- First thought was correlation between a variety of responses and year / program  / demography
    - Correlation within responses
- Histograms will be your friend

---

## Codebook read-in


- There are probably a few ways to hit this, there may be encoding in an Excel tab
- Could Parse out the text from the PDF
   - Regex but will be messy
    - Gemini / Claude API for one-shot NLP?

After spending some time fiddling with a Regex search,  I think 

### **Scope is Graduate School Leadership**

- I think start by scoping the potential types of questions:

  - Financial
  - Academic by:
    - Program
    - Year
  - Demographic?  Doesn't seem there are strong demographic indicators in this set
    - Response bias / survey bias



## **Primary Analysis**

> This analysis focuses on the use of AI by graduate program. The goal is to understand how AI impacts perception of learning outcomes.

- The primary dimension is Program, with splits by Social Sciencees / 

In [1]:
#  --- Using env_finance see environment.yml for quick package installation
#  (Altair is the visualization layer for this section: `pip install altair` if missing)

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import altair as alt
import statsmodels.api as sm
from IPython.display import display, Markdown


In [2]:
# === Display helper: consistent styling for inline tables


# --- Global Pandas Styler table styles for a consistent look

_TABLE_STYLES = [
    {"selector": "caption",
     "props": [("caption-side", "top"), ("text-align", "left"),
               ("font-size", "1rem"), ("font-weight", "600"),
               ("padding", "0.3rem 0 0.5rem 0"), ("color", "#1a1a2e")]},
    {"selector": "th",
     "props": [("background-color", "#5C7077"), ("text-align", "left"),
               ("font-weight", "600"), ("padding", "0.35rem 0.6rem"),
               ("border-bottom", "2px solid #d4d4dc")]},
    {"selector": "td",
     "props": [("padding", "0.3rem 0.6rem"),
               ("border-bottom", "1px solid #ededf2")]},
]

def show_table(df, title=None, hide_index=False, precision=2, max_cols=None):
    """Return a consistently styled Styler for a DataFrame.

    title      : optional caption above the table
    hide_index : drop the index column (use when the index is just a RangeIndex)
    precision  : float decimal places
    max_cols   : cap columns for wide frames (Styler renders all columns otherwise)
    """
    view, note = df, ""
    if max_cols is not None and view.shape[1] > max_cols:
        view = view.iloc[:, :max_cols]
        note = f"  (first {max_cols} of {df.shape[1]} columns)"
    styler = (
        view.style
            .format(precision=precision, na_rep="—", thousands=",")
            .set_table_styles(_TABLE_STYLES)
    )
    if title is not None:
        styler = styler.set_caption(title + note)
    elif note:
        styler = styler.set_caption(note.strip())
    if hide_index:
        styler = styler.hide(axis="index")
    return styler


# --- Text wrapping helper for matplotlib

from matplotlib.text import Text

def wrap_to_axis(text, ax, fontsize=None, fontweight=None, pad_frac=0.98):
    """Break a title onto new lines when it would exceed the axis width."""
    fig = ax.figure
    fig.canvas.draw()                                  # need real geometry to measure
    renderer = fig.canvas.get_renderer()
    max_px = ax.get_window_extent(renderer).width * pad_frac
    probe = Text(0, 0, "", figure=fig, fontsize=fontsize, fontweight=fontweight)
    def w(s):
        probe.set_text(s); return probe.get_window_extent(renderer).width
    lines, cur = [], ""
    for word in text.split():
        trial = word if not cur else cur + " " + word
        if w(trial) <= max_px or not cur:
            cur = trial
        else:
            lines.append(cur); cur = word
    if cur: lines.append(cur)
    return "\n".join(lines)


In [3]:
# Data read-ins and directory setup

directory = os.getcwd()

raw_survey = pd.read_excel(directory + "/CU Boulder SERU dataset_sample.xlsx")

# === Data Dictionary

data_dictionary = pd.read_csv(directory + "/seru_data_dictionary.csv")

data_dictionary[["prefix","sub_question", "suffix"]] = data_dictionary['survey_column'].str.split("_", n=2, expand=True)
data_dictionary['section_code']  = data_dictionary['sub_question'].str[:4]
data_dictionary['sub_question']  = data_dictionary['sub_question'].str[4:]

## **EDA**
#### The following blocks:

 - Conduct basic EDA _(exploratory data analysis)_ using standard methods:
    - `.head()`  - First 10 entries of the dataset
    - `.describe()` - Summary statistics for each variable (column)
    - `.info()` - Data types / metadata  / general information
    - `.shape()`  - The number of rows and columns as `(Rows x Columns)`

 - Look for common data hiccups like:
    - Missing values
    - Incorrect / broken variable types
    - 

---

In [4]:
raw_survey.head()

,PROGRESS,DURATION,FINISHED,CONSENT,GS0101_GSYPPROGNAME,GS0102_GSYPSPECPROG,GS0103B_GSYPSPECSTRT,GS0104_GSYPTAKECRSE_r23,GS0104_GSYPWORKDISS_r23,GS0104_GSYPDEFDDISS_r23,...,CIP_CODE1,COLLEGE_CODE1,COLLEGE_NAME1,LEVEL_GRAD,LOCATION,PROGRAM_CODE1,PROGRAM_TEXT1,YEAR,TERM1,CIP_CODE2010
0,100,1668,True,Agree,Program name (Seed file),"Educ Foundations, Pol & Prac",Fourth year,Checked,Not Checked,Not Checked,...,130901,EDUC,School of Education,1,Boulder,EFPP,"Educ Foundations, Pol & Prac",2021.0,Summer,130901.0
1,100,1677,True,Agree,Program name (Seed file),Physics,Sixth year or above,Not Checked,Checked,Not Checked,...,400801,ARSC,College of Arts & Sciences,3,Boulder,PHYS,Physics,2019.0,Fall,400801.0
2,100,94906,True,Agree,Program name (Seed file),Electrical Engineering,First year,Checked,Not Checked,Not Checked,...,141001,ENGR,College of Engr & Applied Sci,3,Boulder,EEEN,Electrical Engineering,2024.0,Fall,141001.0
3,100,1334,True,Agree,Program name (Seed file),Astrophysical & Planetary Sci,Third year,Checked,Checked,Not Checked,...,400202,ARSC,College of Arts & Sciences,3,Boulder,ASPS,Astrophysical & Planetary Sci,2022.0,Fall,400202.0
4,22,541,False,Agree,Program name (Seed file),Electrical Engineering,Fourth year,Not Checked,Checked,Not Checked,...,141001,ENGR,College of Engr & Applied Sci,3,Boulder,EEEN,Electrical Engineering,2021.0,Fall,141001.0


In [5]:
# --- Tabular description of the survey dataset

display(Markdown("## First 5 rows of the survey dataset"))
display(show_table(raw_survey.head(), max_cols=12))
display(Markdown("---"))
display(Markdown("## Survey Summary Statistics"))
display(show_table(raw_survey.describe()))
display(Markdown("---"))


## First 5 rows of the survey dataset

,PROGRESS,DURATION,FINISHED,CONSENT,GS0101_GSYPPROGNAME,GS0102_GSYPSPECPROG,GS0103B_GSYPSPECSTRT,GS0104_GSYPTAKECRSE_r23,GS0104_GSYPWORKDISS_r23,GS0104_GSYPDEFDDISS_r23,GS0104_GSYPHAVEINTN_r23,GS0104_GSYPHAVERESD_r23
0,100,"1,668",True,Agree,Program name (Seed file),"Educ Foundations, Pol & Prac",Fourth year,Checked,Not Checked,Not Checked,Not Checked,Not Checked
1,100,"1,677",True,Agree,Program name (Seed file),Physics,Sixth year or above,Not Checked,Checked,Not Checked,Not Checked,Not Checked
2,100,"94,906",True,Agree,Program name (Seed file),Electrical Engineering,First year,Checked,Not Checked,Not Checked,Checked,Not Checked
3,100,"1,334",True,Agree,Program name (Seed file),Astrophysical & Planetary Sci,Third year,Checked,Checked,Not Checked,Not Checked,Checked
4,22,541,False,Agree,Program name (Seed file),Electrical Engineering,Fourth year,Not Checked,Checked,Not Checked,Not Checked,Not Checked


---

## Survey Summary Statistics

,PROGRESS,DURATION,GS1208_GSDMCHLD0TO1_2,GS1208_GSDMCHLD0TO1_3,CIP_CODE1,LEVEL_GRAD,YEAR,CIP_CODE2010
count,"1,387.00","1,387.00",101.00,101.00,"1,387.00","1,387.00","1,206.00","1,207.00"
mean,87.73,"233,743.01",0.11,0.59,"258,498.77",2.53,"2,022.14","257,793.39"
std,28.09,"718,568.51",1.07,1.21,"142,780.81",0.85,2.11,"143,460.78"
min,11.00,39.00,-1.00,-1.00,"30,103.00",1.00,"2,005.00","30,103.00"
25%,100.00,"1,119.00",-1.00,-1.00,"140,401.00",3.00,"2,021.00","140,401.00"
50%,100.00,"1,680.00",0.00,1.00,"230,101.00",3.00,"2,023.00","230,101.00"
75%,100.00,"5,640.00",1.00,2.00,"400,601.00",3.00,"2,024.00","400,601.00"
max,100.00,"4,496,931.00",3.00,3.00,"540,101.00",4.00,"2,025.00","540,101.00"


---

In [6]:
display(show_table(
    pd.DataFrame({
        "Data Type": raw_survey.dtypes.astype(str),
        "Unique Values": raw_survey.nunique(),
        "Sample": [raw_survey[c].dropna().iloc[0] if raw_survey[c].notna().any() else None
                   for c in raw_survey.columns],
    }).rename_axis("Variable").reset_index(),
    title="Column Types", hide_index=True,
))

Variable,Data Type,Unique Values,Sample
PROGRESS,int64,35,100
DURATION,int64,"1,204","1,668"
FINISHED,bool,2,True
CONSENT,str,1,Agree
GS0101_GSYPPROGNAME,str,2,Program name (Seed file)
GS0102_GSYPSPECPROG,str,76,"Educ Foundations, Pol & Prac"
GS0103B_GSYPSPECSTRT,object,7,Fourth year
GS0104_GSYPTAKECRSE_r23,str,2,Checked
GS0104_GSYPWORKDISS_r23,str,2,Not Checked
GS0104_GSYPDEFDDISS_r23,str,2,Not Checked


In [7]:
# --- General Info about the survey dataset

display(Markdown("## Survey Info"))
raw_survey.info() 
display(Markdown("---"))
display(Markdown(f"## Survey Shape: `{raw_survey.shape}`"))
display(Markdown("---"))
#display(Markdown(f"## Survey Variables:"))
#display(Markdown("\n".join(f"- `{c}`" for c in raw_survey.columns)))


## Survey Info

<class 'pandas.DataFrame'>
RangeIndex: 1387 entries, 0 to 1386
Columns: 289 entries, PROGRESS to CIP_CODE2010
dtypes: bool(1), float64(4), int64(4), object(252), str(28)
memory usage: 3.4+ MB


---

## Survey Shape: `(1387, 289)`

---

In [8]:
# --- Looking for potential missing values in the survey dataset

# Using Pandas indexing to identify missing values in the survey dataset
missing_data = pd.DataFrame({
    "Data Type": raw_survey.dtypes,
    "Quantity Missing": raw_survey.isnull().sum(),
})

missing_data = missing_data.rename_axis("Variable").reset_index()
missing_data = missing_data[missing_data["Quantity Missing"] > 0]

display(Markdown("## Missing Values in the Survey Dataset"))
display(show_table(missing_data, hide_index=True))
display(Markdown("---"))

## Missing Values in the Survey Dataset

Variable,Data Type,Quantity Missing
GS1101_GSOSQLTYINST,object,41
GS1101_GSOSCRSEAVLB,object,41
GS1101_GSOSQLTYADVS,object,41
GS1101_GSOSAVLBADVS,object,41
GS1101_GSOSKNWLGAIN,object,41
GS1101_GSOSFNCLSPPT,object,41
GS1101_GSOSQLTYEQPT,object,41
GS1101_GSOSQLTYOFIT,object,41
GS1101_GSOSQLTYPRDV,object,41
GS1101_GSOSQLTYLIBR,object,41


---

In [9]:
# === Basic Cleaning 

# Drop where all values are missing
raw_survey = raw_survey.dropna(how='all')

ai_use_codes = ["GS1301_GSGAIDAY",
                "GS1303_GSGAIPDC",
                "GS1303_GSGAIAUC",
                "GS1303_GSGAIGPT",
                "GS1303_GSGAIGURW",
                "GS1303_GSGAIGR",
                "GS1303_GSGAIEPDR"]

raw_survey = raw_survey[(raw_survey[ai_use_codes].notnull().any(axis=1))]



In [10]:
raw_survey[ai_use_codes].head()

,GS1301_GSGAIDAY,GS1303_GSGAIPDC,GS1303_GSGAIAUC,GS1303_GSGAIGPT,GS1303_GSGAIGURW,GS1303_GSGAIGR,GS1303_GSGAIEPDR
0,Several times per year,Strongly disagree,Agree,NaN,Strongly disagree,Agree,Agree
1,Daily,Strongly disagree,Strongly disagree,NaN,Strongly disagree,Strongly disagree,Agree
2,Several times per year,Agree,Agree,NaN,Agree,Strongly agree,Agree
3,Several times per month,Strongly agree,Strongly agree,NaN,Strongly agree,Strongly agree,Strongly agree
5,Several times per month,Agree,Agree,Agree,Agree,Agree,Agree


In [11]:
# === Visualization style — CU Boulder palette + Altair theme
# Pivoted from the earlier matplotlib rcParams; every chart below is Altair.

CU_GOLD, CU_GRAY, CU_BLACK = "#CFB87C", "#565A5C", "#000000"
CU_CATEGORY = [CU_GOLD, CU_GRAY, CU_BLACK, "#9EA2A2", "#7A6F52"]

_cu_config = {
    "view":   {"stroke": None},
    "axis":   {"grid": False, "labelColor": CU_BLACK, "titleColor": CU_BLACK,
               "labelFontSize": 11, "titleFontSize": 12, "labelLimit": 500},
    "title":  {"anchor": "start", "color": CU_BLACK, "fontSize": 15},
    "legend": {"labelFontSize": 11, "titleFontSize": 12},
    "range":  {"category": CU_CATEGORY},
}

# Theme registration moved API location in Altair 5.5; this works on 5.0 through 6.x.
try:
    @alt.theme.register("cu", enable=True)
    def _cu_theme(): return {"config": _cu_config}
except AttributeError:
    alt.themes.register("cu", lambda: {"config": _cu_config})
    alt.themes.enable("cu")

alt.data_transformers.disable_max_rows()   # melted survey subset can exceed the 5k default


DataTransformerRegistry.enable('default')

## **Generative AI — Summary Visualizations & Tables**

First-pass views of the three AI batteries: overall **frequency** of use (`GS1301`),
the **purposes** students use it for (`GS1302`, multiselect), and **attitudes / guidance**
(`GS1303`, agree–disagree). Then one cut by college, and a coverage check over `YEAR`
before any trend is trusted.


In [12]:
# Ordered response scales, so charts sort by meaning rather than alphabetically
FREQ_ORDER   = ["Never", "Several times per year", "Several times per month",
                "Several times per week", "Daily"]
LIKERT_ORDER = ["Strongly disagree", "Disagree", "Agree", "Strongly agree"]
weekly_levels = ["Several times per week", "Daily"]

# Human-readable item text from the data dictionary (fall back to the raw code)
item_text = data_dictionary.set_index("survey_column")["item_text"].to_dict()
label = lambda code: item_text.get(code, code)

# GS1302 purposes are a multiselect battery not included in ai_use_codes; pull them here
purpose_cols  = [c for c in raw_survey.columns if c.startswith("GS1302_GSGGAI")]
attitude_cols = ["GS1303_GSGAIPDC", "GS1303_GSGAIAUC", "GS1303_GSGAIGPT",
                 "GS1303_GSGAIGURW", "GS1303_GSGAIGR", "GS1303_GSGAIEPDR"]

ai = raw_survey.dropna(subset=["GS1301_GSGAIDAY"]).copy()
ai["YEAR"] = ai["YEAR"].astype("Int64")

LEVEL_LABELS = {1: "Research master's", 2: "Professional master's",
                3: "Research doctorate", 4: "Professional doctorate"}
unmapped = set(raw_survey["LEVEL_GRAD"].dropna()) - set(LEVEL_LABELS)
assert not unmapped, f"unmapped LEVEL_GRAD values: {unmapped}"

In [13]:
# Frequency of AI use — the headline adoption metric

freq = (raw_survey["GS1301_GSGAIDAY"].value_counts()
        .reindex(FREQ_ORDER).fillna(0).astype(int)
        .rename_axis("Frequency").reset_index(name="Respondents"))
freq["Percent"] = 100 * freq["Respondents"] / freq["Respondents"].sum()

display(show_table(freq, title="How often graduate students use generative AI", hide_index=True))

alt.Chart(freq).mark_bar(color=CU_GOLD).encode(
    x=alt.X("Frequency:N", sort=FREQ_ORDER, title=None, axis=alt.Axis(labelAngle=-25)),
    y=alt.Y("Percent:Q", title="% of respondents"),
    tooltip=["Frequency", "Respondents", alt.Tooltip("Percent:Q", format=".1f")],
).properties(width=1120, height=640, title="Generative AI use frequency")


Frequency,Respondents,Percent
Never,278,23.62
Several times per year,303,25.74
Several times per month,265,22.51
Several times per week,208,17.67
Daily,123,10.45


alt.Chart(...)

In [14]:
# Attitudes & guidance (agree–disagree). 100% stacked, ordered by share agreeing.
att = raw_survey[attitude_cols].melt(var_name="code", value_name="Response").dropna()
att["Item"] = att["code"].map(label)
att["rank"] = att["Response"].map({r: i for i, r in enumerate(LIKERT_ORDER)})

item_order = (att.assign(agree=att["Response"].isin(["Agree", "Strongly agree"]))
              .groupby("Item")["agree"].mean().sort_values().index.tolist())

LIKERT_COLORS = ["#7A2E2E", "#C6A0A0", "#D8C79A", CU_GOLD]   # disagree -> agree

display(show_table(
    att.assign(Agree=att["Response"].isin(["Agree", "Strongly agree"]))
       .groupby("Item").agg(**{"% Agree": ("Agree", "mean"), "n": ("Agree", "size")})
       .assign(**{"% Agree": lambda d: 100 * d["% Agree"]})
       .sort_values("% Agree", ascending=False).reset_index(),
    title="Agreement with AI statements (top-two-box)", hide_index=True))

alt.Chart(att).mark_bar().encode(
    y=alt.Y("Item:N", sort=item_order, title=None),
    x=alt.X("count():Q", stack="normalize", title="Share of responses", axis=alt.Axis(format="%")),
    color=alt.Color("Response:N", sort=LIKERT_ORDER,
                    scale=alt.Scale(domain=LIKERT_ORDER, range=LIKERT_COLORS),
                    legend=alt.Legend(orient="top", title=None)),
    order=alt.Order("rank:Q"),
    tooltip=["Item", "Response", alt.Tooltip("count():Q", title="n")],
).properties(width=1200, height=50 * len(item_order), title="Attitudes toward generative AI")

Item,% Agree,n
I understand when it is appropriate to use AI to complete my coursework,85.00,"1,187"
I understand how AI generates responses,84.50,"1,187"
I understand how to create effective AI prompts that produce desired responses,73.38,"1,187"
My professors have discussed when it is appropriate to use AI to complete my coursework,65.71,"1,187"
I have received guidance from my graduate program about appropriate uses of AI in my research and writing,43.86,"1,181"
If Have you held a paid position that required you to teach students of the university since you beg... = Yes I have received guidance from my graduate program about appropriate uses of AI in my teaching,40.26,837


alt.Chart(...)

In [15]:
# What students use AI for (multiselect) — share among those who saw the battery (AI users)
answered = raw_survey[purpose_cols].notna().any(axis=1)

use_label = {
        "GS1302_GSGGAI_1" : "To research a topic",
        "GS1302_GSGGAI_4" : "To brainstorm ideas for a project or paper",
        "GS1302_GSGGAI_5" : "To draft a paper, report, or presentation",
        "GS1302_GSGGAI_6" : "To revise a paper, report, or presentation",
        "GS1302_GSGGAI_7" : "To draft responses for assignments",
        "GS1302_GSGGAI_8"  : "To revise responses for assignments", 
        "GS1302_GSGGAI_9" : "To generate programming code",
        "GS1302_GSGGAI_10" : "To revise/debug programming code",
        "GS1302_GSGGAI_11" : "To prepare for exams or certifications",
        "GS1302_GSGGAI_12" : "To assist with translations for academic or professional work",
        "GS1302_GSGGAI_999" : "Other (Specified)"} # <- Qualtrics other value
purpose = (raw_survey.loc[answered, purpose_cols].eq("Checked").mean()
           .mul(100).rename("Percent").rename_axis("code").reset_index())
purpose["Purpose"] = purpose["code"].map(use_label)

display(show_table(purpose[["Purpose", "Percent"]].sort_values("Percent", ascending=False),
                   title="Purposes of AI use (share of AI users)", hide_index=True))

alt.Chart(purpose).mark_bar(color=CU_GOLD).encode(
    y=alt.Y("Purpose:N", sort="-x", title=None),
    x=alt.X("Percent:Q", title="% of AI users"),
    tooltip=["Purpose", alt.Tooltip("Percent:Q", format=".1f")],
).properties(width=560, height=28 * len(purpose), title="What students use generative AI for")


Purpose,Percent
To revise/debug programming code,56.33
To research a topic,46.97
To generate programming code,45.87
To brainstorm ideas for a project or paper,43.89
"To revise a paper, report, or presentation",31.13
To assist with translations for academic or professional work,19.80
"To draft a paper, report, or presentation",12.10
Other (Specified),11.00
To prepare for exams or certifications,10.89
To revise responses for assignments,9.68


alt.Chart(...)

In [16]:
# Frequent use (weekly or more) by college, against the institution baseline

by_college = (ai.assign(weekly=lambda d: d["GS1301_GSGAIDAY"].isin(weekly_levels))
              .groupby("COLLEGE_NAME1")["weekly"].mean().mul(100)
              .rename("Percent").reset_index())

institution = ai["GS1301_GSGAIDAY"].isin(weekly_levels).mean() * 100

bars = alt.Chart(by_college).mark_bar(color=CU_GOLD).encode(
    y=alt.Y("COLLEGE_NAME1:N", sort="-x", title=None),
    x=alt.X("Percent:Q", title="% using AI weekly or more"),
    tooltip=["COLLEGE_NAME1", alt.Tooltip("Percent:Q", format=".1f")])

baseline = alt.Chart(pd.DataFrame({"x": [institution]})).mark_rule(
    color=CU_BLACK, strokeDash=[4, 4]).encode(x="x:Q")

(bars + baseline).properties(width=1120, height=68 * len(by_college),
    title="Frequent AI use by college  (dashed line = institution overall)")


alt.LayerChart(...)

Okay before I start doing my own pass at this file, I'm going to have you do another big revision. Right now it all runs wonderfully, and seems mostly accurate. I just want you to take one more pass with the following goals:







Clean up code for clarity - It's pretty good right now, but dense. Some of the density is necessary, some of it is probably excessive. I generally try to avoid creating  intermediate variables (things that are keys / helpers that kind of thing, let me know if that makes sense) when I code. Sometimes it is helpful, but it also makes it hard downstream to understand what is a global tool that exists for necessary things like data cleaning and transformation, and what only exists to build a certain figure or table.



Do a quick validity pass to make sure that there aren't any glaring issues.



Organization - If it can be improved / cleaned up that would be appreciated



Simplicity - Don't break anything, but where possible, if there's a simpler way to achieve the same outcome, do so



Data management - This is the biggest one, and it feels inadequate right now. I'm going to attach an export of all columns and head values so you can actually see what is in the survey data. There is a lot in there, and ideally, the first parts of this would organize it. Whether that is splitting information into sub- data frames or something else 

In [17]:
# Before trusting a time trend: does the AI subset actually span multiple survey years?
ai_rows = raw_survey.dropna(subset=["GS1301_GSGAIDAY"]).copy()
ai_rows["YEAR"] = ai_rows["YEAR"].astype("Int64")

coverage = ai_rows.groupby("YEAR").size().rename("Respondents").reset_index()
display(show_table(coverage, title="AI-question respondents by YEAR", hide_index=True))

# Trend in frequent use (weekly+). If only one or two years appear above, read this as a
# snapshot / slope, not a trend — the cross-sectional views are the real story.
trend = (ai_rows.assign(weekly=lambda d: d["GS1301_GSGAIDAY"].isin(weekly_levels))
         .groupby("YEAR")["weekly"].mean().mul(100).rename("Percent").reset_index())

alt.Chart(trend).mark_line(point=True, color=CU_GOLD).encode(
    x=alt.X("YEAR:O", title=None),
    y=alt.Y("Percent:Q", title="% using AI weekly or more"),
    tooltip=["YEAR", alt.Tooltip("Percent:Q", format=".1f")],
).properties(width=1120, height=640, title="Frequent AI use over time")


YEAR,Respondents
"2,005",1
"2,006",1
"2,010",1
"2,012",2
"2,013",1
"2,014",1
"2,015",9
"2,016",5
"2,017",4
"2,018",14


alt.Chart(...)

In [18]:
# The one conditioning check worth having in your pocket: is a college gap really a
# degree-level (LEVEL_GRAD) composition effect? Confirm these labels against your dictionary.

by_level = (raw_survey.assign(
        weekly=lambda d: d["GS1301_GSGAIDAY"].isin(weekly_levels),
        Level=lambda d: d["LEVEL_GRAD"].map(LEVEL_LABELS))
    .groupby("Level")["weekly"].mean().mul(100).rename("Percent").reset_index())

alt.Chart(by_level).mark_bar(color=CU_GOLD).encode(
    x=alt.X("Level:N", title=None),
    y=alt.Y("Percent:Q", title="% using AI weekly or more"),
    tooltip=["Level", alt.Tooltip("Percent:Q", format=".1f")],
).properties(width=720, height=600, title="Frequent AI use by degree level")


/tmp/ipykernel_1534034/1627245368.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  by_level = (raw_survey.assign(
/tmp/ipykernel_1534034/1627245368.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  by_level = (raw_survey.assign(


alt.Chart(...)